# SmolVLA ファインチューニング実験：短時間 vs 長時間（ブドウ色分けタスク）

**目的**：同じデータ（グレープ色分け49エピソード）を使い、SmolVLAのLoRAファインチューニングを短時間・長時間の2パターンで試して、学習量が精度にどう効くかを比較する。

**事前準備**
1. `so101_grape_sort_dataset.zip`（ACT学習の時に使ったのと同じもの）がGoogleドライブ `MyDrive/so101/` にアップロード済みであることを確認
2. **短時間ラン**：T4ランタイムでOK
3. **長時間ラン**：A100ランタイムに切り替えてから実行推奨（同じ時間でこなせるステップ数が大きく変わるため）

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ZIP = "/content/drive/MyDrive/so101/so101_grape_sort_dataset.zip"

!mkdir -p /content/dataset
!unzip -q "$DRIVE_ZIP" -d /content/dataset
!ls /content/dataset

In [ ]:
# lerobot本体 + PEFT(LoRA)ライブラリ + SmolVLAのプロセッサに必要なnum2words
# torchaoはColab既存のバージョン(0.10.0)が古すぎてPEFTの要求(0.16.0以上)を満たさないため明示的にアップグレード
!pip install -q "lerobot[dataset,training]==0.6.0" accelerate peft num2words "torchao>=0.16.0"

In [ ]:
import glob

candidates = glob.glob("/content/dataset/so101_grape_sort_*")
assert len(candidates) == 1, f"データセットフォルダが1つに絞れません: {candidates}"
DATASET_ROOT = candidates[0]
print("DATASET_ROOT =", DATASET_ROOT)

EPISODES = "[" + ",".join(str(i) for i in range(49)) + "]"  # episode 49(受け渡し失敗)を除外

## 短時間ラン（T4でOK、目安5000ステップ）

In [ ]:
!lerobot-train \
  --dataset.repo_id=local/so101_grape_sort \
  --dataset.root="$DATASET_ROOT" \
  --dataset.episodes="$EPISODES" \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.load_vlm_weights=true \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --peft.r=16 \
  --output_dir=/content/drive/MyDrive/so101/train_output/smolvla_grape_sort_short \
  --job_name=smolvla_grape_sort_short \
  --steps=5000 \
  --batch_size=8 \
  --save_freq=1000 \
  --log_freq=50 \
  --wandb.enable=false

## 長時間ラン（★A100ランタイムに切り替えてから実行推奨、目安50000ステップ）

実行前に「ランタイム→ランタイムのタイプを変更」でA100に切り替えてください（切り替えると一度セッションがリセットされるので、上のセルから順に再実行が必要です）。

In [ ]:
!lerobot-train \
  --dataset.repo_id=local/so101_grape_sort \
  --dataset.root="$DATASET_ROOT" \
  --dataset.episodes="$EPISODES" \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.load_vlm_weights=true \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --peft.r=16 \
  --output_dir=/content/drive/MyDrive/so101/train_output/smolvla_grape_sort_long \
  --job_name=smolvla_grape_sort_long \
  --steps=50000 \
  --batch_size=8 \
  --save_freq=5000 \
  --log_freq=100 \
  --wandb.enable=false

In [ ]:
# 完了確認
!echo "--- short ---"; ls -la /content/drive/MyDrive/so101/train_output/smolvla_grape_sort_short/checkpoints
!echo "--- long ---"; ls -la /content/drive/MyDrive/so101/train_output/smolvla_grape_sort_long/checkpoints